In [ ]:
!pip install -q sentence-transformers faiss-gpu-cu12 networkx pandas numpy scikit-learn tqdm

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value = user_secrets.get_secret("hf_token")

In [ ]:
from huggingface_hub import login
login(token=secret_value)

In [ ]:
import json
import pickle
import re
from collections import defaultdict

import faiss
import networkx as nx
import numpy as np
import torch
from sentence_transformers import CrossEncoder, SentenceTransformer
from tqdm import tqdm

In [ ]:
MODEL_NAME = "BAAI/bge-m3"
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"
EMBED_BATCH = 32
MAX_RERANK_CANDIDATES = 200

LAW_RELATION_EDGE_TYPES = {
    "REFERENCES",
    "AMENDS",
    "REPLACES",
    "DETAILS"
}
LAW_RELATION_WEIGHTS = {
    "DETAILS": 0.9,
    "REFERENCES": 0.9,
    "AMENDS": 0.9,
    "REPLACES": 0.9,
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def load_queries_jsonl(path):
    queries = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            item = json.loads(line)
            queries[str(item["_id"])] = item.get("text", "").strip()
    return queries


def load_qrels_jsonl(path):
    qrels = defaultdict(set)
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            item = json.loads(line)
            if item.get("score", 0) > 0:
                qrels[str(item["query-id"])].add(str(item["corpus-id"]))
    return dict(qrels)


def load_knowledge_graph(path):
    path = str(path)
    if path.endswith(".graphml"):
        return nx.read_graphml(path)

    if path.endswith(".json"):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        graph = nx.DiGraph()
        for node_data in data.get("nodes", []):
            attrs = dict(node_data)
            node_id = str(attrs.pop("id"))
            graph.add_node(node_id, **attrs)

        for edge_data in data.get("edges", []):
            attrs = dict(edge_data)
            source = str(attrs.pop("source"))
            target = str(attrs.pop("target"))
            graph.add_edge(source, target, **attrs)

        return graph

    raise ValueError(f"Unsupported graph format: {path}")


def load_paragraph_vector_store(embeddings_file, faiss_index_file):
    with open(embeddings_file, "rb") as f:
        embeddings_data = pickle.load(f)

    embeddings_dict = embeddings_data["embeddings_dict"]
    paragraph_ids = [str(pid) for pid in embeddings_data["paragraph_ids"]]
    index = faiss.read_index(str(faiss_index_file))

    if index.ntotal != len(paragraph_ids):
        raise ValueError(
            f"FAISS index has {index.ntotal} vectors but pickle has "
            f"{len(paragraph_ids)} paragraph ids"
        )

    return embeddings_dict, paragraph_ids, index


def embed_queries(model, texts, batch_size=EMBED_BATCH):
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        show_progress_bar=True,
    )
    embeddings = embeddings.astype(np.float32)
    faiss.normalize_L2(embeddings)
    return embeddings


def extract_section_id(paragraph_id):
    match = re.match(r"^(.+?)_para_\d+$", str(paragraph_id))
    if match:
        return match.group(1)
    return str(paragraph_id)


def extract_law_id(section_id):
    match = re.match(r"^(.+?)\+\d+$", str(section_id))
    if match:
        return match.group(1)
    return str(section_id)


def get_edge_attr(graph, source, target):
    data = graph.get_edge_data(source, target, default={})
    if data and "edge_type" not in data and all(isinstance(v, dict) for v in data.values()):
        return next(iter(data.values()))
    return data or {}


def get_edge_type(edge_attr):
    return edge_attr.get("edge_type") or edge_attr.get("relationship_type") or edge_attr.get("label")


def safe_float(value, default=0.0):
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def update_score(scores, key, score):
    if score > scores.get(key, float("-inf")):
        scores[key] = score


def paragraph_text(embeddings_dict, paragraph_id):
    info = embeddings_dict.get(paragraph_id, {})
    parts = [
        info.get("section_title", ""),
        info.get("label", ""),
        info.get("content", ""),
    ]
    return "\n".join(part.strip() for part in parts if isinstance(part, str) and part.strip())


def build_paragraph_maps(paragraph_ids):
    section_to_paragraphs = defaultdict(list)
    law_to_sections = defaultdict(list)
    seen_law_sections = defaultdict(set)

    for paragraph_id in paragraph_ids:
        section_id = extract_section_id(paragraph_id)
        law_id = extract_law_id(section_id)

        section_to_paragraphs[section_id].append(paragraph_id)
        if section_id not in seen_law_sections[law_id]:
            law_to_sections[law_id].append(section_id)
            seen_law_sections[law_id].add(section_id)

    return dict(section_to_paragraphs), dict(law_to_sections)


def get_parent_law(graph, section_id):
    if section_id in graph:
        node_type = graph.nodes[section_id].get("node_type")
        if node_type == "LAW":
            return section_id

        predecessors = graph.predecessors(section_id) if hasattr(graph, "predecessors") else graph.neighbors(section_id)
        for pred in predecessors:
            edge_type = get_edge_type(get_edge_attr(graph, pred, section_id))
            if graph.nodes[pred].get("node_type") == "LAW" and edge_type == "HAS_SECTION":
                return pred

    return extract_law_id(section_id)


def iter_law_sections(graph, law_id, law_to_sections):
    if law_id in law_to_sections:
        yield from law_to_sections[law_id]
        return

    if law_id not in graph:
        return

    successors = graph.successors(law_id) if hasattr(graph, "successors") else graph.neighbors(law_id)
    for child in successors:
        edge_type = get_edge_type(get_edge_attr(graph, law_id, child))
        if graph.nodes[child].get("node_type") == "SECTION" and edge_type == "HAS_SECTION":
            yield child


def iter_related_laws(graph, law_id):
    if law_id not in graph:
        return

    successors = graph.successors(law_id) if hasattr(graph, "successors") else graph.neighbors(law_id)
    for target in successors:
        edge_attr = get_edge_attr(graph, law_id, target)
        edge_type = get_edge_type(edge_attr)
        if graph.nodes[target].get("node_type") != "LAW":
            continue
        if edge_type and edge_type not in LAW_RELATION_EDGE_TYPES:
            continue

        yield target, LAW_RELATION_WEIGHTS.get(edge_type, 0.5)

    if not hasattr(graph, "predecessors"):
        return

    for source in graph.predecessors(law_id):
        edge_attr = get_edge_attr(graph, source, law_id)
        edge_type = get_edge_type(edge_attr)
        if graph.nodes[source].get("node_type") != "LAW":
            continue
        if edge_type and edge_type not in LAW_RELATION_EDGE_TYPES:
            continue

        yield source, LAW_RELATION_WEIGHTS.get(edge_type, 0.5)


def add_section_paragraphs(candidate_scores, section_to_paragraphs, section_id, score):
    for paragraph_id in section_to_paragraphs.get(section_id, []):
        update_score(candidate_scores, paragraph_id, score)


def ranked_unique_sections(paragraph_ids):
    section_ids = []
    seen = set()
    for paragraph_id in paragraph_ids:
        section_id = extract_section_id(paragraph_id)
        if section_id in seen:
            continue
        section_ids.append(section_id)
        seen.add(section_id)
    return section_ids


def paragraph_kg2rag_retrieve_and_rerank(
    index,
    graph,
    paragraph_ids,
    embeddings_dict,
    section_to_paragraphs,
    law_to_sections,
    q_emb_single,
    query_text,
    reranker,
    top_k_seeds=30,
    m_hops=1,
    top_k_final=10,
    max_rerank_candidates=MAX_RERANK_CANDIDATES,
):
    q_emb = np.array([q_emb_single], dtype=np.float32)
    faiss.normalize_L2(q_emb)
    distances, indices = index.search(q_emb, top_k_seeds)

    seed_paragraph_ids = []
    paragraph_similarities = {}

    for idx, score in zip(indices[0], distances[0]):
        if idx == -1:
            continue

        paragraph_id = paragraph_ids[idx]
        seed_paragraph_ids.append(paragraph_id)
        paragraph_similarities[paragraph_id] = float(score)

    if not seed_paragraph_ids:
        return [], [], [], []

    candidate_scores = {}
    law_scores = {}

    for paragraph_id, similarity in paragraph_similarities.items():
        update_score(candidate_scores, paragraph_id, similarity)

        section_id = extract_section_id(paragraph_id)
        law_id = get_parent_law(graph, section_id)
        update_score(law_scores, law_id, similarity)

    frontier = dict(law_scores)
    visited_laws = set(frontier.keys())

    for _ in range(max(0, m_hops)):
        next_frontier = {}

        for law_id, law_score in frontier.items():
            for related_law_id, relation_weight in iter_related_laws(graph, law_id):
                related_score = law_score * relation_weight

                for section_id in iter_law_sections(graph, related_law_id, law_to_sections):
                    add_section_paragraphs(
                        candidate_scores,
                        section_to_paragraphs,
                        section_id,
                        related_score,
                    )

                if related_law_id not in visited_laws:
                    update_score(next_frontier, related_law_id, related_score)
                    visited_laws.add(related_law_id)

        frontier = next_frontier

        if not frontier:
            break

    candidate_paragraph_ids = sorted(
        candidate_scores,
        key=lambda paragraph_id: candidate_scores[paragraph_id],
        reverse=True,
    )[:max_rerank_candidates]

    before_rerank_paragraph_ids = candidate_paragraph_ids
    before_rerank_section_ids = ranked_unique_sections(before_rerank_paragraph_ids)

    cross_inputs = []
    valid_paragraph_ids = []

    for paragraph_id in candidate_paragraph_ids:
        text = paragraph_text(embeddings_dict, paragraph_id)

        if not text:
            continue

        cross_inputs.append([query_text, text])
        valid_paragraph_ids.append(paragraph_id)

    if not cross_inputs:
        return (
            before_rerank_section_ids[:top_k_final],
            before_rerank_paragraph_ids[:max_rerank_candidates],
            before_rerank_section_ids[:top_k_final],
            before_rerank_paragraph_ids[:max_rerank_candidates],
        )

    rerank_scores = reranker.predict(cross_inputs, batch_size=256)

    reranked = sorted(
        zip(valid_paragraph_ids, rerank_scores),
        key=lambda item: item[1],
        reverse=True,
    )

    after_rerank_paragraph_ids = [paragraph_id for paragraph_id, _ in reranked]
    after_rerank_section_ids = ranked_unique_sections(after_rerank_paragraph_ids)

    return (
        before_rerank_section_ids[:top_k_final],
        before_rerank_paragraph_ids[:max_rerank_candidates],
        after_rerank_section_ids[:top_k_final],
        after_rerank_paragraph_ids[:max_rerank_candidates],
    )

def reciprocal_rank_at_k(ranked_ids, gt_ids, k):
    for rank, item_id in enumerate(ranked_ids[:k], start=1):
        if item_id in gt_ids:
            return 1.0 / rank
    return 0.0


def init_metrics(k_list):
    return {
        "hits": {k: 0 for k in k_list},
        "recalls": {k: 0.0 for k in k_list},
        "precisions": {k: 0.0 for k in k_list},
        "f1_scores": {k: 0.0 for k in k_list},
        "mrrs": {k: 0.0 for k in k_list},
    }


def update_metrics(metrics, ranked_section_ids, gt_section_ids, k_list):
    for k in k_list:
        preds = set(ranked_section_ids[:k])
        correct = preds.intersection(gt_section_ids)
        num_correct = len(correct)

        if num_correct > 0:
            metrics["hits"][k] += 1

        recall = num_correct / len(gt_section_ids)
        precision = num_correct / k
        mrr = reciprocal_rank_at_k(ranked_section_ids, gt_section_ids, k)

        metrics["recalls"][k] += recall
        metrics["precisions"][k] += precision
        metrics["mrrs"][k] += mrr

        if recall + precision > 0:
            metrics["f1_scores"][k] += 2 * recall * precision / (recall + precision)


def print_metrics_table(title, metrics, num_queries, k_list):
    print("\n" + "=" * 96)
    print(title)
    print("=" * 96)

    if num_queries == 0:
        print("No queries with ground truth to evaluate.")
        return

    print(
        f"{'K':>3} | "
        f"{'Recall':>7} | "
        f"{'Precision':>9} | "
        f"{'F1':>7} | "
        f"{'Hit':>7} | "
        f"{'MRR':>7}"
    )
    print("-" * 96)

    for k in k_list:
        print(
            f"{k:>3d} | "
            f"{metrics['recalls'][k] / num_queries:7.4f} | "
            f"{metrics['precisions'][k] / num_queries:9.4f} | "
            f"{metrics['f1_scores'][k] / num_queries:7.4f} | "
            f"{metrics['hits'][k] / num_queries:7.4f} | "
            f"{metrics['mrrs'][k] / num_queries:7.4f}"
        )


def run_kg2rag_evaluation(
    index,
    graph,
    paragraph_ids,
    embeddings_dict,
    section_to_paragraphs,
    law_to_sections,
    eval_queries,
    eval_q_embs,
    eval_qrels,
    reranker,
):
    k_list = [1, 5, 10]

    before_metrics = init_metrics(k_list)
    after_metrics = init_metrics(k_list)

    num_queries = 0

    for i in tqdm(range(len(eval_queries)), desc="Evaluating Paragraph KG^2RAG"):
        query_id, query_text = eval_queries[i]
        q_emb = eval_q_embs[i]
        gt_section_ids = eval_qrels[query_id]

        num_queries += 1

        (
            before_rerank_section_ids,
            _,
            after_rerank_section_ids,
            _,
        ) = paragraph_kg2rag_retrieve_and_rerank(
            index,
            graph,
            paragraph_ids,
            embeddings_dict,
            section_to_paragraphs,
            law_to_sections,
            q_emb,
            query_text,
            reranker,
        )

        update_metrics(
            before_metrics,
            before_rerank_section_ids,
            gt_section_ids,
            k_list,
        )

        update_metrics(
            after_metrics,
            after_rerank_section_ids,
            gt_section_ids,
            k_list,
        )

    print_metrics_table(
        "PARAGRAPH KG^2RAG BEFORE CROSS-ENCODER RERANKER",
        before_metrics,
        num_queries,
        k_list,
    )

    print_metrics_table(
        "PARAGRAPH KG^2RAG AFTER CROSS-ENCODER RERANKER",
        after_metrics,
        num_queries,
        k_list,
    )

def main():
    embeddings_file = "/kaggle/input/notebooks/minhvb10/qa-retrieval-embed/paragraph_embeddings.pkl"
    faiss_index_file = "/kaggle/input/notebooks/minhvb10/qa-retrieval-embed/paragraph_embeddings.faiss"
    queries_file = "/kaggle/input/datasets/minhvb10/zalo-dataset/queries_preprocessed_v2.jsonl"
    test_file = "/kaggle/input/datasets/minhvb10/zalo-dataset/test.jsonl"
    graph_file = "/kaggle/input/datasets/minhvb10/zalo-dataset/legal_kg_complete.json"

    print("1. Loading paragraph vector store and KG...")
    embeddings_dict, paragraph_ids, index = load_paragraph_vector_store(
        embeddings_file,
        faiss_index_file,
    )
    section_to_paragraphs, law_to_sections = build_paragraph_maps(paragraph_ids)

    graph = load_knowledge_graph(graph_file)
    queries_dict = load_queries_jsonl(queries_file)
    qrels_dict = load_qrels_jsonl(test_file)

    eval_queries = [
        (query_id, queries_dict[query_id])
        for query_id in qrels_dict
        if query_id in queries_dict
    ]

    print(f"Loaded {len(paragraph_ids)} paragraphs from {embeddings_file}")
    print(f"Loaded FAISS index from {faiss_index_file}")
    print(f"Loaded KG: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")
    print(f"Loaded {len(queries_dict)} queries; evaluating {len(eval_queries)} queries with qrels")

    print("\n2. Loading embedding model for queries...")
    embedding_model = SentenceTransformer(MODEL_NAME, device=DEVICE)
    if DEVICE == "cuda":
        embedding_model.half()

    eval_query_texts = [query_text for _, query_text in eval_queries]
    print("Embedding queries...")
    eval_q_embs = embed_queries(embedding_model, eval_query_texts)

    print("\n3. Loading CrossEncoder reranker...")
    reranker = CrossEncoder(RERANK_MODEL, max_length=1024, device=DEVICE)
    if DEVICE == "cuda":
        reranker.model.half()

    print("\n4. Starting evaluation...")
    run_kg2rag_evaluation(
        index,
        graph,
        paragraph_ids,
        embeddings_dict,
        section_to_paragraphs,
        law_to_sections,
        eval_queries,
        eval_q_embs,
        qrels_dict,
        reranker,
    )


if __name__ == "__main__":
    main()